# Análisis en profundidad — clase **G4C** en `partition/`

**LUT de mask (referencia para segmentación):** `[25:74)→GG3`, `[75:174)→GG4`, `[175:)→GG5`; definida como `_MASK_LUT` en la primera celda de código (huecos en grises 74 y 174 → NC).

**Kernel:** usa el intérprete del entorno del proyecto (p. ej. `prostata_env`) si `pandas`/`openpyxl` fallan al leer los `.xlsx`.

Los Excel de cada fold (`Train.xlsx`, `Test.xlsx`) tienen una etiqueta **principal** one-hot (`NC`, `G3`, `G4`, `G5`) y una column adicional **`G4C`**. **G4C no es una quinta clase paralela:** es un **subtipo de GG4** —el patrón Gleason 4 **cribiforme** versus otros patrones 4 (p. ej. acinar/fusionado). En un flujo de anotación como el de los scripts **`TrainCribiform`** (u homólogos), **only intervienen rows ya clasificadas como G4**; después se marca si ese parche es cribiforme (**G4C=1**) o patrón 4 no cribiforme (**G4C=0**). Así, G4C presupone **G4=1**.

En **segmentación con 4 clases** (NC, GG3, GG4, GG5), los parches G4 siguen siendo **GG4** tanto si G4C=1 como si G4C=0; G4C sirve como metadata o para análisis secundario.

Este notebook:
- Recorre **todos** los `.xlsx` bajo `partition/`.
- Cuantifica **G4C** y su co-ocurrencia con `G3`/`G4`/`G5`/`NC`.
- Localiza **slides y pacientes** donde predominan parches G4C.
- Cruza optionalmente con `wsi_labels.xlsx` (Gleason por lámina) si está disponible.

**Comprobación:** el crosstab y la celda siguiente verifican que `G4C=1` only aparece con `G4=1` (subtipo dentro de GG4).

In [22]:
from __future__ import annotations

from pathlib import Path

import numpy as np
import pandas as pd

try:
    from IPython.display import display
except ImportError:
    display = print  # fuera de Jupyter


def find_project_root() -> Path:
    for p in [Path.cwd(), *Path.cwd().parents]:
        if (p / "partition").is_dir():
            return p
    raise FileNotFoundError(
        "Not found 'partition/'. Abre el notebook desde la raíz del proyecto SICAPv2 "
        "o asigna BASE = Path(r'...') manualmente en la siguiente celda."
    )


BASE = find_project_root()
PARTITION = BASE / "partition"
WSI_LABELS = BASE / "wsi_labels.xlsx"

_MASK_LUT = np.zeros(256, dtype=np.int64)
_MASK_LUT[25:74] = 1
_MASK_LUT[75:174] = 2
_MASK_LUT[175:] = 3

print("BASE:", BASE)

BASE: c:\Users\Aleix\OneDrive - Universitat Politècnica de Catalunya\Escritorio\UNI\TFG\Recerca primers datasets\SicapV2\SICAPv2


In [23]:
def list_partition_excels() -> list[tuple[Path, str, str]]:
    """Lista (ruta, etiqueta_fold, Train|Test)."""
    out: list[tuple[Path, str, str]] = []
    patterns = [
        ("Validation/*/Train.xlsx", "Train"),
        ("Validation/*/Test.xlsx", "Test"),
        ("Test/Train.xlsx", "Train"),
        ("Test/Test.xlsx", "Test"),
    ]
    for pat, split in patterns:
        for p in sorted(PARTITION.glob(pat)):
            rel = p.relative_to(PARTITION)
            if rel.parts[0] == "Validation" and len(rel.parts) >= 3:
                fold = f"Validation/{rel.parts[1]}"
            elif rel.parts[0] == "Test":
                fold = "Test"
            else:
                fold = "/".join(rel.parts[:-1])
            out.append((p, fold, split))
    return out


excel_files = list_partition_excels()
print(f"Files Excel: {len(excel_files)}")
for path, fold, sp in excel_files:
    print(f"  [{fold}] {sp}: {path.name}")

Ficheros Excel: 10
  [Validation/Val1] Train: Train.xlsx
  [Validation/Val2] Train: Train.xlsx
  [Validation/Val3] Train: Train.xlsx
  [Validation/Val4] Train: Train.xlsx
  [Validation/Val1] Test: Test.xlsx
  [Validation/Val2] Test: Test.xlsx
  [Validation/Val3] Test: Test.xlsx
  [Validation/Val4] Test: Test.xlsx
  [Test] Train: Train.xlsx
  [Test] Test: Test.xlsx


In [24]:
def load_all_partition_rows() -> pd.DataFrame:
    rows = []
    for path, fold, split in excel_files:
        df = pd.read_excel(path)
        df["_source_file"] = path.name
        df["_partition_relpath"] = str(path.relative_to(BASE))
        df["_fold"] = fold
        df["_split"] = split
        rows.append(df)
    return pd.concat(rows, ignore_index=True)


df_all = load_all_partition_rows()
print("Filas totales (puede haber duplicados de image_name entre folds):", len(df_all))
print("Columnas:", df_all.columns.tolist())

Filas totales (puede haber duplicados de image_name entre folds): 51917
Columnas: ['image_name', 'NC', 'G3', 'G4', 'G5', 'G4C', '_source_file', '_partition_relpath', '_fold', '_split']


## De dónde sale el total de rows (~51k)

El `DataFrame` **concatena los 10 Excel** (cada uno es un `Train.xlsx` o `Test.xlsx` bajo una carpeta). Hay **cuatro folds** `Validation/Val1` … `Val4` más la carpeta `Test/` final: en validación cruzada, **los mismos parches** (`image_name`) se listan otra vez en cada fold con la misma anotación. Por eso:

`rows totales ≈ n_rows_por_Excel × n_files`, y **no** es el número de parches únicos del dataset.

La siguiente tabla desglosa rows por **ruta relativa** dentro del proyecto.

In [25]:
by_dir = (
    df_all.groupby("_partition_relpath", as_index=False)
    .agg(n_rows=("image_name", "count"))
    .sort_values("_partition_relpath")
    .reset_index(drop=True)
)
by_dir["pct_del_total"] = (100 * by_dir["n_rows"] / len(df_all)).round(2)
display(by_dir)

# Vista compacta: fold × Train/Test
pivot = df_all.pivot_table(
    index="_fold", columns="_split", values="image_name", aggfunc="count", fill_value=0
)
print("\nFilas por fold (row) y split (column):")
display(pivot)

n_unique = df_all["image_name"].nunique()
n_files = df_all["_partition_relpath"].nunique()
print(f"\n--- Resumen ---")
print(f"Filas totales (suma de los {n_files} Excel): {len(df_all):,}")
print(f"`image_name` únicos en todo el concat: {n_unique:,}")
print(f"Filas medias por file: {len(df_all) / n_files:.0f}")
print(
    f"Repetición: cada parche aparece ~{len(df_all) / n_unique:.1f} veces en promedio "
    f"(mismos parches en varios folds Excel)."
)

,_partition_relpath,n_filas,pct_del_total
0,partition\Test\Test.xlsx,2122,4.09
1,partition\Test\Train.xlsx,9959,19.18
2,partition\Validation\Val1\Test.xlsx,2487,4.79
3,partition\Validation\Val1\Train.xlsx,7472,14.39
4,partition\Validation\Val2\Test.xlsx,2166,4.17
5,partition\Validation\Val2\Train.xlsx,7793,15.01
6,partition\Validation\Val3\Test.xlsx,1793,3.45
7,partition\Validation\Val3\Train.xlsx,8166,15.73
8,partition\Validation\Val4\Test.xlsx,3513,6.77
9,partition\Validation\Val4\Train.xlsx,6446,12.42



Filas por fold (fila) y split (columna):


_split,Test,Train
_fold,,
Test,2122,9959
Validation/Val1,2487,7472
Validation/Val2,2166,7793
Validation/Val3,1793,8166
Validation/Val4,3513,6446



--- Resumen ---
Filas totales (suma de los 10 Excel): 51,917
`image_name` únicos en todo el concat: 12,081
Filas medias por fichero: 5192
Repetición: cada parche aparece ~4.3 veces en promedio (mismos parches en varios folds Excel).


## Esquema de columns e integridad one-hot

- **Clase principal (una sola):** `NC`, `G3`, `G4`, `G5` — exactamente un `1` por row.
- **Subtipo only si G4:** `G4C` es un flag **adicional** (`0`/`1`) definido **únicamente cuando la clase principal es G4** (mismo criterio que en pipelines tipo `TrainCribiform`: el conjunto de trabajo para cribiforme son las rows **G4**; luego se distingue cribiforme vs no cribiforme).

In [26]:
df_all[df_all['image_name'] == '18B0001510J_Block_Region_2_0_54_xini_60831_yini_62018.jpg']

,image_name,NC,G3,G4,G5,G4C,_source_file,_partition_relpath,_fold,_split
2845,18B0001510J_Block_Region_2_0_54_xini_60831_yin...,0,0,0,1,0,Train.xlsx,partition\Validation\Val1\Train.xlsx,Validation/Val1,Train
9856,18B0001510J_Block_Region_2_0_54_xini_60831_yin...,0,0,0,1,0,Train.xlsx,partition\Validation\Val2\Train.xlsx,Validation/Val2,Train
18588,18B0001510J_Block_Region_2_0_54_xini_60831_yin...,0,0,0,1,0,Train.xlsx,partition\Validation\Val3\Train.xlsx,Validation/Val3,Train
37501,18B0001510J_Block_Region_2_0_54_xini_60831_yin...,0,0,0,1,0,Test.xlsx,partition\Validation\Val4\Test.xlsx,Validation/Val4,Test
43523,18B0001510J_Block_Region_2_0_54_xini_60831_yin...,0,0,0,1,0,Train.xlsx,partition\Test\Train.xlsx,Test,Train


In [27]:
df_all[(df_all['G5'] == 1) & (df_all['_fold'] == 'Validation/Val1')]

,image_name,NC,G3,G4,G5,G4C,_source_file,_partition_relpath,_fold,_split
0,16B0001851_Block_Region_1_0_0_xini_6803_yini_5...,0,0,0,1,0,Train.xlsx,partition\Validation\Val1\Train.xlsx,Validation/Val1,Train
1,16B0001851_Block_Region_1_0_1_xini_7827_yini_5...,0,0,0,1,0,Train.xlsx,partition\Validation\Val1\Train.xlsx,Validation/Val1,Train
2,16B0001851_Block_Region_1_0_2_xini_8851_yini_5...,0,0,0,1,0,Train.xlsx,partition\Validation\Val1\Train.xlsx,Validation/Val1,Train
3,16B0001851_Block_Region_1_0_3_xini_9875_yini_5...,0,0,0,1,0,Train.xlsx,partition\Validation\Val1\Train.xlsx,Validation/Val1,Train
4,16B0001851_Block_Region_1_1_0_xini_6803_yini_6...,0,0,0,1,0,Train.xlsx,partition\Validation\Val1\Train.xlsx,Validation/Val1,Train
...,...,...,...,...,...,...,...,...,...,...
31665,18B0006179I_Block_Region_1_3_46_xini_58832_yin...,0,0,0,1,0,Test.xlsx,partition\Validation\Val1\Test.xlsx,Validation/Val1,Test
31666,18B0006179I_Block_Region_1_3_47_xini_59856_yin...,0,0,0,1,0,Test.xlsx,partition\Validation\Val1\Test.xlsx,Validation/Val1,Test
31667,18B0006179I_Block_Region_1_3_48_xini_60880_yin...,0,0,0,1,0,Test.xlsx,partition\Validation\Val1\Test.xlsx,Validation/Val1,Test
31668,18B0006179I_Block_Region_1_3_49_xini_61904_yin...,0,0,0,1,0,Test.xlsx,partition\Validation\Val1\Test.xlsx,Validation/Val1,Test


In [28]:
base_cols = [c for c in ["NC", "G3", "G4", "G5"] if c in df_all.columns]
has_g4c = "G4C" in df_all.columns

if not has_g4c:
    raise ValueError("No hay column G4C en los Excel; revisa la versión del dataset.")

df_all["_main_sum"] = df_all[base_cols].sum(axis=1)
df_all["_main_label"] = df_all[base_cols].idxmax(axis=1)

print("Distribución etiqueta principal (NC/G3/G4/G5 por idxmax):")
print(df_all["_main_label"].value_counts())
print()
print("Filas donde la suma NC+G3+G4+G5 no es 1:")
bad = df_all["_main_sum"] != 1
print(bad.value_counts())
if bad.any():
    display(df_all.loc[bad, ["image_name", "NC", "G3", "G4", "G5", "G4C", "_main_sum", "_fold"]].head(20))

Distribución etiqueta principal (NC/G3/G4/G5 por idxmax):
_main_label
NC    19509
G4    19058
G3     9538
G5     3812
Name: count, dtype: int64

Filas donde la suma NC+G3+G4+G5 no es 1:
_main_sum
False    51917
Name: count, dtype: int64


## Prevalencia de **G4C** (subtipo dentro de **G4**) y cruces con la clase principal

In [29]:
g4c = df_all["G4C"].fillna(0).astype(int)
n_g4c = int((g4c == 1).sum())
print(f"Parches con G4C=1: {n_g4c:,} ({100 * n_g4c / len(df_all):.2f}% del total de rows cargadas)")
print(f"Parches con G4C=0: {int((g4c == 0).sum()):,}")

# Cruce G4C x etiqueta principal
ct = pd.crosstab(df_all["_main_label"], g4c, margins=True)
ct.columns = ["G4C=0", "G4C=1", "All"]
print("\nCrosstab: etiqueta principal (rows) vs G4C (columns)")
print(ct)
print("\nInterpretación: G4C=1 only aparece en rows con etiqueta principal G4 (subtipo cribiforme dentro de GG4).")
print("NC/G3/G5 no tienen G4C=1 en estos datos — coherente con TrainCribiform (only rows G4).")

Parches con G4C=1: 3,235 (6.23% del total de filas cargadas)
Parches con G4C=0: 48,682

Crosstab: etiqueta principal (filas) vs G4C (columnas)
             G4C=0  G4C=1    All
_main_label                     
G3            9538      0   9538
G4           15823   3235  19058
G5            3812      0   3812
NC           19509      0  19509
All          48682   3235  51917

Interpretación: G4C=1 solo aparece en filas con etiqueta principal G4 (subtipo cribiforme dentro de GG4).
NC/G3/G5 no tienen G4C=1 en estos datos — coherente con TrainCribiform (solo filas G4).


In [30]:
# Jerarquía G4 → G4C (como en TrainCribiform: only rows con G4; luego cribiforme sí/no)
m = g4c == 1
sub = df_all.loc[m, ["NC", "G3", "G4", "G5", "G4C"]].copy()
print("Entre rows con G4C=1 (subtipo cribiforme):")
print("  G4=1:", int((sub["G4"] == 1).sum()), "  (esperado: todas)")
print("  G4=0:", int((sub["G4"] == 0).sum()), "  (debe ser 0 si G4C ⊆ G4)")

only_g4c = df_all.loc[m & (df_all["G4"] == 0)]
print(f"\nFilas con G4C=1 pero G4=0 (inconsistentes): {len(only_g4c)}")
if len(only_g4c):
    display(only_g4c[["image_name", "NC", "G3", "G4", "G5", "G4C", "_fold", "_split"]].head(30))

# Dentro de todas las rows con clase principal G4: fracción cribiforme (G4C=1) vs no
g4_rows = df_all["_main_label"] == "G4"
n_g4 = int(g4_rows.sum())
n_g4_crib = int((g4_rows & (g4c == 1)).sum())
n_g4_noncrib = n_g4 - n_g4_crib
print(f"\n--- Subdivisión del conjunto G4 (=GG4) ---")
print(f"Filas con etiqueta principal G4: {n_g4:,}")
print(f"  · G4C=1 (GG4 cribiforme):     {n_g4_crib:,} ({100 * n_g4_crib / n_g4:.1f}% del G4)")
print(f"  · G4C=0 (GG4 no cribiforme):  {n_g4_noncrib:,} ({100 * n_g4_noncrib / n_g4:.1f}% del G4)")
print("\nEsto coincide con la lógica TrainCribiform: el cribiforme se anota sobre parches ya etiquetados como G4.")

Entre filas con G4C=1 (subtipo cribiforme):
  G4=1: 3235   (esperado: todas)
  G4=0: 0   (debe ser 0 si G4C ⊆ G4)

Filas con G4C=1 pero G4=0 (inconsistentes): 0

--- Subdivisión del conjunto G4 (=GG4) ---
Filas con etiqueta principal G4: 19,058
  · G4C=1 (GG4 cribiforme):     3,235 (17.0% del G4)
  · G4C=0 (GG4 no cribiforme):  15,823 (83.0% del G4)

Esto coincide con la lógica TrainCribiform: el cribiforme se anota sobre parches ya etiquetados como G4.


## Por fold y por slide/paciente

Los nombres de imagen suelen comenzar por un identificador de bloque/slide antes de `_Block_`.

In [31]:
def slide_id_from_image(name: str) -> str:
    s = str(name)
    if "_Block_" in s:
        return s.split("_Block_", 1)[0]
    return s.split("_", 1)[0]


df_all["slide_id"] = df_all["image_name"].map(slide_id_from_image)

g4c_by_fold = (
    df_all.assign(G4C_bin=g4c)
    .groupby(["_fold", "_split"], as_index=False)
    .agg(n=("image_name", "count"), n_g4c=("G4C_bin", "sum"))
)
g4c_by_fold["pct_g4c"] = 100.0 * g4c_by_fold["n_g4c"] / g4c_by_fold["n"]
print("G4C por fold y split:")
display(g4c_by_fold.sort_values(["_fold", "_split"]))

slide_stats = (
    df_all.assign(G4C_bin=g4c)
    .groupby("slide_id")
    .agg(n_patches=("image_name", "count"), n_g4c=("G4C_bin", "sum"))
    .sort_values("n_g4c", ascending=False)
)
slide_stats["pct_g4c"] = 100.0 * slide_stats["n_g4c"] / slide_stats["n_patches"]
print("\nTop 25 slides por número absoluto de parches G4C:")
display(slide_stats.head(25))

G4C por fold y split:


,_fold,_split,n,n_g4c,pct_g4c
0,Test,Test,2122,145,6.833176
1,Test,Train,9959,618,6.205442
2,Validation/Val1,Test,2487,237,9.529554
3,Validation/Val1,Train,7472,381,5.099036
4,Validation/Val2,Test,2166,41,1.892890
5,Validation/Val2,Train,7793,577,7.404081
6,Validation/Val3,Test,1793,126,7.027328
7,Validation/Val3,Train,8166,492,6.024982
8,Validation/Val4,Test,3513,214,6.091660
9,Validation/Val4,Train,6446,404,6.267453



Top 25 slides por número absoluto de parches G4C:


,n_patches,n_g4c,pct_g4c
slide_id,,,
18B0004349G,570,425,74.561404
17B0024636,490,420,85.714286
18B0001510J,895,390,43.575419
18B0004346A,670,370,55.223881
18B0002975B,630,255,40.476190
17B0017511,345,150,43.478261
18B0001971B,170,145,85.294118
18B0006177A,405,135,33.333333
17B0019359,275,115,41.818182


## Opcional: cruce con `wsi_labels.xlsx` (Gleason por lámina)

Si existe, podemos ver si las láminas con muchos parches G4C tienen patrones Gleason concretos a nivel WSI.

In [32]:
if WSI_LABELS.is_file():
    wsi = pd.read_excel(WSI_LABELS)
    print("Columnas wsi_labels:", wsi.columns.tolist())
    id_col = "slide_id" if "slide_id" in wsi.columns else wsi.columns[0]
    gp = "Gleason_primary" if "Gleason_primary" in wsi.columns else None
    gs = "Gleason_secondary" if "Gleason_secondary" in wsi.columns else None

    top_slides = slide_stats.head(40).index.tolist()
    sub_wsi = wsi[wsi[id_col].astype(str).isin(top_slides)]
    cols = [c for c in [id_col, gp, gs] if c]
    print("\nGleason (WSI) para slides con más parches G4C (muestra):")
    display(sub_wsi[cols] if len(cols) > 1 else sub_wsi)

    merged = df_all[["slide_id", "G4C"]].copy()
    merged["G4C"] = merged["G4C"].fillna(0).astype(int)
    slide_g4c_rate = merged.groupby("slide_id")["G4C"].mean().rename("mean_g4c")
    wsi_small = wsi.set_index(id_col)
    if gp and gs:
        wsi_small = wsi_small[[gp, gs]].copy()
        wsi_small["score_str"] = wsi_small[gp].astype(str) + "+" + wsi_small[gs].astype(str)
        joined = wsi_small.join(slide_g4c_rate, how="inner")
        print("\nMedia de G4C por parche vs score Gleason (por slide, only slides en ambos):")
        print(joined.groupby("score_str")["mean_g4c"].agg(["mean", "count"]).sort_values("mean", ascending=False).head(20))
else:
    print(f"Not found {WSI_LABELS}; salta esta sección.")

Columnas wsi_labels: ['slide_id', 'patient_id', 'Gleason_primary', 'Gleason_secondary']

Gleason (WSI) para slides con más parches G4C (muestra):


,slide_id,Gleason_primary,Gleason_secondary
0,16B0001851,4,5
2,16B0003394,3,3
3,16B0006668,5,5
4,16B0006669,5,5
5,16B0006694,4,5
6,16B0006695,4,5
8,16B0008067,4,5
11,16B0022612,3,4
12,16B0022613,4,3
13,16B0022615,4,3



Media de G4C por parche vs score Gleason (por slide, solo slides en ambos):
               mean  count
score_str                 
4+4        0.309128     13
4+5        0.179818     28
4+3        0.042603     23
5+4        0.020408      7
3+4        0.001894     22
0+0        0.000000     36
3+3        0.000000     14
3+5        0.000000      4
5+3        0.000000      1
5+5        0.000000      7


## Deduplicar por `image_name` (mismo parche en varios Excel)

Si un mismo `image_name` aparece en más de un file con la misma row, los conteos anteriores lo cuentan varias veces. Aquí dejamos una row por nombre.

In [33]:
n_before = len(df_all)
df_u = df_all.drop_duplicates(subset=["image_name"], keep="first")
print(f"Filas únicas por image_name: {len(df_u)} (antes {n_before}, duplicados {n_before - len(df_u)})")

g4c_u = df_u["G4C"].fillna(0).astype(int)
print(f"\nCon deduplicación — parches con G4C=1: {int((g4c_u == 1).sum()):,} ({100 * (g4c_u == 1).mean():.2f}%)")

Filas únicas por image_name: 12081 (antes 51917, duplicados 39836)

Con deduplicación — parches con G4C=1: 763 (6.32%)


## Resumen interpretativo

- **G4C (GG4 cribiforme)** es un **subtipo de GG4**, no una clase Gleason distinta de G4. La anotación sigue la misma idea que en **`TrainCribiform`**: el conjunto de entrada son los parches con **G4=1**; sobre ellos se indica si el patrón 4 es **cribiforme** (`G4C=1`) u otro patrón 4 (`G4C=0`).
- En **segmentación a 4 clases**, todos los `G4=1` son **GG4**; la column `G4C` sirve para estratificar o analizar cribiforme vs no cribiforme dentro de GG4.
- Los datos cargados aquí cumplen **G4C ⊆ G4** (no hay `G4C=1` sin `G4=1`), coherente con ese flujo de anotación.